# Guide intelligent du Musée · 06 Recommandation

Évaluer une seule fois sur le test les modèles retenus, puis répondre à la direction : un réseau de neurones vaut-il le coup en production, ou un modèle classique suffit-il ?

Contexte du projet, données et découpage : voir `01_donnees.ipynb`.

## 1. Mise en place

In [ ]:
import math
import platform
import time

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix

from commun import (IMAGENET_MEAN, IMAGENET_STD, MODELS_DIR, UNKNOWN_ARTIST, load_dense, load_images, load_metadata,
                    load_resnet, normalization_stats, set_seed, split_indices, standardize)

set_seed()
meta, CLASSES = load_metadata()
X_all, y_all = load_images(meta)
splits = split_indices(meta)
MEAN, STD = normalization_stats(X_all, splits["train"])

print(f"PyTorch {torch.__version__}, calculs sur CPU")
print(f"{len(meta)} œuvres, {len(CLASSES)} styles ; " + ", ".join(f"{k} : {len(i)}" for k, i in splits.items()))

## 2. Évaluation finale sur le test

Les trois modèles retenus sont rechargés depuis `modeles/`, tels que sauvegardés par `02_baseline`, `03_diagnostic` et `04_resnet`, sans réentraînement ni réglage. Tous les choix ont été faits sur la validation. C'est la seule utilisation du test dans le projet, et aucun modèle n'a été modifié après cette évaluation. Tout est calculé sur CPU, comme sur une borne.

### 2.1 Tableau final

- accuracy de validation et de test, avec l'intervalle de confiance à 95 % sur les 300 œuvres du test ;
- temps d'inférence pour une œuvre, prétraitement compris, médiane sur 100 appels, avec tous les cœurs du processeur puis avec un seul, plus proche d'une borne modeste ;
- taille du modèle, en paramètres et en Mo sur disque ;
- coût d'entraînement : ordre de grandeur pour entraîner une fois le modèle retenu sur un Mac M1. Il est mesuré à part, car rien n'est réentraîné ici : environ 5 s sur CPU pour la régression logistique, 7 s sur CPU pour les 30 époques du réseau dense, et 5 minutes sur le GPU du Mac pour le ResNet (`04_resnet` : 20 époques de tête, puis 7 époques de fine-tuning).

In [ ]:
logreg_path, dense_path, resnet_path = (MODELS_DIR / name for name in
                                         ("regression_logistique.joblib", "reseau_dense.pt", "resnet.pt"))
logreg, dense, resnet = joblib.load(logreg_path), load_dense(dense_path), load_resnet(resnet_path)


def logreg_predict(x: torch.Tensor) -> np.ndarray:
    return logreg.predict(standardize(x, MEAN, STD).flatten(1).numpy())


def dense_predict(x: torch.Tensor) -> np.ndarray:
    return dense(standardize(x, MEAN, STD)).argmax(1).numpy()


def resnet_predict(x: torch.Tensor) -> np.ndarray:
    return resnet(standardize(x, IMAGENET_MEAN, IMAGENET_STD)).argmax(1).numpy()


def predict_split(predict_fn, X: torch.Tensor, batch_size: int = 100) -> np.ndarray:
    return np.concatenate([predict_fn(X[i:i + batch_size]) for i in range(0, len(X), batch_size)])


def latency_ms(predict_fn, x: torch.Tensor, n: int = 100) -> float:
    """Temps médian d'une prédiction pour une seule œuvre uint8 (C, H, W), en millisecondes."""
    predict_fn(x[None])  # premier appel hors mesure
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        predict_fn(x[None])
        times.append(time.perf_counter() - t0)
    return 1000 * float(np.median(times))


MODELS = {
    "régression logistique": (logreg_predict, logreg.coef_.size + logreg.intercept_.size, logreg_path, "≈ 5 s, CPU"),
    "réseau dense": (dense_predict, sum(p.numel() for p in dense.parameters()), dense_path, "≈ 7 s, CPU"),
    "ResNet-18 fine-tuné": (resnet_predict, sum(p.numel() for p in resnet.parameters()), resnet_path, "≈ 5 min, GPU"),
}
X_val, y_val = X_all[splits["val"]], y_all[splits["val"]].numpy()
X_test, y_test = X_all[splits["test"]], y_all[splits["test"]].numpy()
n_threads = torch.get_num_threads()
rows, test_preds = [], {}
with torch.no_grad():
    for name, (predict_fn, n_params, path, training) in MODELS.items():
        acc_val = (predict_split(predict_fn, X_val) == y_val).mean()
        test_preds[name] = predict_split(predict_fn, X_test)
        acc_test = (test_preds[name] == y_test).mean()
        ms_all = latency_ms(predict_fn, X_test[0])
        torch.set_num_threads(1)
        ms_one = latency_ms(predict_fn, X_test[0])
        torch.set_num_threads(n_threads)
        rows.append({"modele": name, "acc_val": acc_val, "acc_test": acc_test,
                     "ic95_test": 1.96 * math.sqrt(acc_test * (1 - acc_test) / len(y_test)),
                     f"ms_{n_threads}_coeurs": ms_all, "ms_1_coeur": ms_one,
                     "parametres": n_params, "taille_mo": path.stat().st_size / 1e6, "entrainement": training})

final = pd.DataFrame(rows).set_index("modele")
print(f"Mesuré sur CPU, {n_threads} cœurs puis 1 : {platform.platform()}")
final.round({"acc_val": 3, "acc_test": 3, "ic95_test": 3, f"ms_{n_threads}_coeurs": 2, "ms_1_coeur": 2, "taille_mo": 1})

- **Sur le test, le ResNet reste loin devant** : 0,750 ± 0,049, contre 0,420 ± 0,056 pour la régression logistique et 0,400 ± 0,055 pour le réseau dense. Les intervalles de confiance ne se recouvrent pas.
- **Le réseau dense n'apporte rien** : il n'est pas plus précis que la régression logistique, et il est plus lent et plus lourd qu'elle.
- **Le ResNet reste rapide sur CPU** : quelques dizaines de millisecondes par œuvre, même sur un seul cœur, pour 44,8 Mo. Ces temps varient d'une exécution à l'autre.

### 2.2 Par style

Accuracy de test par style pour les trois modèles, puis matrice de confusion du ResNet : une ligne par style réel, une colonne par style prédit.

In [ ]:
per_style = pd.DataFrame({name: [(preds[y_test == c] == c).mean() for c in range(len(CLASSES))]
                          for name, preds in test_preds.items()}, index=CLASSES)

cm = pd.DataFrame(confusion_matrix(y_test, test_preds["ResNet-18 fine-tuné"]), index=CLASSES, columns=CLASSES)
cm.index.name, cm.columns.name = "style réel", "style prédit"
errors = cm.where(~np.eye(len(CLASSES), dtype=bool)).stack().dropna().astype(int)
print(f"ResNet : {errors.sum()} œuvres mal classées sur {len(y_test)}")
print("Erreurs les plus fréquentes :", ", ".join(f"{a} pris pour {b} {n}" for (a, b), n in errors.nlargest(3).items()))
print(per_style.round(3).to_string(), "\n")
cm

- **Seul le ResNet reconnaît les cinq styles** : son accuracy par style va de 0,567 à 0,833. Les deux autres modèles tombent à 0,150 pour l'Impressionism (régression logistique) et à 0,183 pour le Cubism (réseau dense).
- **Le Cubism reste le style le plus difficile** : 0,567, et 13 œuvres cubistes sur 60 sont prises pour du Pop Art.
- **Le Baroque et l'Impressionism se confondent aussi** : 10 œuvres baroques prises pour de l'Impressionism, et 9 dans l'autre sens.

### 2.3 Écart entre la validation et le test

Le ResNet perd plus de points que les deux autres modèles entre la validation et le test. Le découpage ne garantit la séparation des peintres que pour les artistes nommés : le détail suivant reprend les prédictions du test, sans rien réévaluer ni modifier.

In [ ]:
for name, row in final.iterrows():
    print(f"{name} : {row['acc_val'] - row['acc_test']:.3f} de moins sur le test que sur la validation")

test_meta = meta.iloc[splits["test"]].reset_index(drop=True)
test_meta = test_meta.assign(artiste=np.where(test_meta["artist"] == UNKNOWN_ARTIST, "inconnu", "nommé"),
                             correct=test_preds["ResNet-18 fine-tuné"] == y_test)
for kind, group in test_meta.groupby("artiste")["correct"]:
    print(f"ResNet, artiste {kind} : {group.sum()} œuvres bien classées sur {len(group)}, accuracy {group.mean():.3f}")
print("Artistes nommés du test :", test_meta.loc[test_meta["artiste"] == "nommé", "artist"].value_counts().to_dict())
test_meta.groupby(["label", "artiste"])["correct"].agg(["size", "mean"]).round(3)

- **L'écart est propre au ResNet** : 0,087 entre la validation et le test, contre 0,007 et 0,010 pour les deux autres modèles.
- **Il ne vient pas des artistes** : 0,743 pour les 35 œuvres d'artistes nommés, 0,751 pour les 265 œuvres d'artistes inconnus. Pour l'Ukiyo-e, 0,765 contre 0,808, sur de petits effectifs.
- **L'explication la plus probable est le choix sur la validation** : l'arrêt précoce et la décision de garder le fine-tuning s'y sont faits, ce qui rend 0,837 optimiste, et le hasard d'un échantillon de 300 œuvres s'y ajoute. Ce n'est qu'une hypothèse, que ces données ne permettent pas de prouver. C'est pour cette raison que le chiffre à annoncer à la direction est celui du test : **0,750**.

## 3. Recommandation

**Oui, un réseau de neurones vaut le coup en production pour les deux fonctions du guide, à condition de partir de modèles pré-entraînés.** Entraîner un réseau de zéro sur nos images, comme le réseau dense, n'apporte rien. Réutiliser un réseau pré-entraîné apporte presque tout le gain, pour un coût faible sur une borne et sans matériel à acheter.

### 3.1 Reconnaître une œuvre : le ResNet-18 fine-tuné

- **Précision** : 0,750 sur le test, contre 0,420 pour la régression logistique. Le guide donne le bon style trois fois sur quatre, au lieu de deux fois sur cinq.
- **Borne** : quelques dizaines de millisecondes par œuvre sur un seul cœur d'un Mac M1. Sur un CPU dix fois plus lent, la réponse arriverait encore en moins d'une demi-seconde. 44,8 Mo de poids, et aucun GPU en production.
- **Équipe et budget** : le modèle s'entraîne une fois, en environ 5 minutes sur le GPU d'un Mac. Il n'y a rien à entraîner sur la borne.
- **Modèles écartés** : la régression logistique est la plus légère et la plus rapide, mais se trompe près de six fois sur dix, ce qui est inutilisable pour un guide. Le réseau dense n'est ni plus précis ni plus léger qu'elle.
- **Présentation au visiteur** : le guide se trompe une fois sur quatre. Il doit donc présenter un style probable, pas une certitude, surtout entre le Cubism et le Pop Art.

### 3.2 Répondre aux questions : les embeddings de phrases

- **Précision** : 0,911 des questions de test trouvent une œuvre attendue parmi les 5 premières proposées, contre 0,622 avec TF-IDF (`05_questions`, section 5). Le gain vient surtout des paraphrases, auxquelles TF-IDF ne répond jamais.
- **Borne** : une dizaine de millisecondes par question sur CPU, 90,853 Mo de poids.
- **Équipe et budget** : aucun entraînement. Le modèle est utilisé tel quel, et les descriptions ne sont encodées qu'une fois.
- **Amélioration conseillée** : ajouter la correspondance exacte des noms d'artistes, que TF-IDF retrouve et que les embeddings ratent parfois (Kuniyoshi, Murillo).

### 3.3 Sur une borne

Les deux modèles représentent environ 136 Mo de poids, sans compter PyTorch, qu'il faut aussi installer. Chacun répond en moins d'un dixième de seconde sur un seul cœur du processeur de test. Avant un déploiement, il reste à mesurer ces temps sur le matériel réel des bornes et à tester la reconnaissance sur de vraies photos (section 4).

## 4. Limites

- **Artistes inconnus** : toutes les œuvres Impressionism et Pop_Art sont attribuées à « Unknown Artist » (`01_donnees`, section 3). Pour ces deux styles, le découpage ne peut pas garantir qu'un même peintre reste dans un seul split.
- **Genres peu renseignés** : 1 229 œuvres sur 1 500 (82 %) ont le genre « Unknown Genre » (`01_donnees`, section 3), ce qui appauvrit les descriptions textuelles du jalon `m4`.
- **Reproductions numériques, pas photos** : les images de WikiArt sont des reproductions propres, alors qu'un visiteur photographie un tableau de biais, avec des reflets et sous l'éclairage d'une salle. Les scores de reconnaissance sont sans doute optimistes : c'est le premier test à faire avant un déploiement.
- **Petit jeu de test** : 300 œuvres seulement, d'où un intervalle de confiance de ± 0,049 pour le ResNet.
- **Cinq styles seulement** : 1 500 œuvres en 128 pixels. Le guide ne reconnaît pas le genre, et ajouter un style demande de réentraîner le modèle.
- **Temps mesurés sur un Mac M1** : une borne modeste sera plus lente, et la mesure sur un seul cœur n'en est qu'une approximation.
- **Licence de WikiArt inconnue** : en production, le modèle devrait être réentraîné sur des images dont le musée a les droits, idéalement celles de sa collection.
- **Questions en anglais, écrites par l'équipe** : `all-MiniLM-L6-v2` est entraîné sur de l'anglais, et un guide en français demanderait un modèle multilingue. Le jeu de 45 questions est aussi écrit par ceux qui ont rédigé les descriptions.
- **Contenu des images non décrit** : « une mer déchaînée » ou « des portraits d'enfants » restent sans réponse, car les métadonnées ne disent pas ce que montre une œuvre. Il faudrait décrire chaque image, par exemple avec un modèle qui relie textes et images comme CLIP.